In [1]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf

In [2]:
df = pd.read_excel('./datasets/pxr_descriptors.xlsx')
df

,Name,SMILES,MW,AMW,Sv,Se,Sp,Si,Mv,Me,...,s2_numAroBonds,s3_numAroBonds,s4_numAroBonds,s34_size,s34_relSize,s34_phSize,s34_phRelSize,chiralMoment,chiralPhMoment,Endocrine Disrupting-PXR
0,ID0001,CCOP(=S)(OCC)Oc1ccc(cc1)N(=O)=O,291.29,9.102812,20.3938,32.9745,21.9375,36.0954,0.637306,1.030453,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,positive
1,ID0002,COC(C)C(NC(=O)C(CC(C)C)C(CCC(F)(F)F)N(O)C=O)C(...,482.59,7.911311,36.3718,63.0286,37.4151,70.7921,0.596259,1.033256,...,0.0,1.25,3.75,27.750000,0.867188,8.000000,0.250000,32.925681,10.239996,positive
2,ID0003,CC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C,332.53,5.938036,31.5732,55.1195,34.5459,63.2714,0.563807,0.984277,...,0.0,0.00,0.00,19.437500,0.809896,2.312500,0.096354,19.026438,2.499118,positive
3,ID0004,CC12CCC3C(CCC4=CC(=O)CCC34C)C2CCC1O,288.47,5.887143,27.8048,48.0250,30.5686,55.2316,0.567445,0.980102,...,0.0,0.00,0.00,16.833333,0.801587,1.416667,0.067460,17.069366,1.609476,positive
4,ID0005,CN1C(=O)N(C(=O)C=C1C(F)(F)F)c1ccc(c(c1)C(=O)OC...,474.85,9.497000,33.5770,52.8652,33.0228,57.3681,0.671540,1.057304,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2362,ID2363,CC(=O)OCC(COC(C)=O)OC(C)=O,218.23,7.525172,16.9764,30.1490,17.0568,33.1628,0.585393,1.039621,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,negative
2363,ID2364,CC(C)C(C)C,86.20,4.310000,9.6876,19.1852,11.3298,22.9064,0.484380,0.959260,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,negative
2364,ID2365,O=C1NC(=O)c2cc(ccc12)N(=O)=O,192.14,10.674444,13.4288,19.3964,12.5908,20.2494,0.746044,1.077578,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,negative
2365,ID2366,ClC(Cl)(Cl)Cl,153.81,30.762000,5.3636,6.0620,5.9544,5.6064,1.072720,1.212400,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,negative


In [3]:
df_smiles = df[['Name', 'SMILES']]

In [4]:
work_dir = 'PXR_OPT'

In [ ]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_smiles,
    smiles_col="SMILES",
    id_col="Name",
    keywords="PM7 PRECISE GEO-OK",
    workdir=work_dir,
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=6,
)

In [12]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, error_message, duration_s]
Index: []


In [ ]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir=work_dir,
    overwrite=True,
    timeout=900,
    max_workers=14,
)

In [8]:
results[results['arc_file'].isna()]#.to_excel('twice_failed_molecules.xlsx')
results[results['arc_file'].isna()]['compound_id']

12    1065-49-2
Name: compound_id, dtype: str

In [10]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir=work_dir,
    overwrite=True,
    max_workers=6,
)

In [11]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []


In [19]:
from pathlib import Path
import shutil

source_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\mopac_jobs")
target_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\sdf_all")

target_dir.mkdir(exist_ok=True)

for sdf_file in source_dir.rglob("*.sdf"):
    shutil.copy2(sdf_file, target_dir / sdf_file.name)